# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and display summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s and structure for this dataset.

In [ ]:
# List all RecordSets and their @ids (unique identifiers)
print("Available Record Sets (with '@id' and name):")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']}   name: {rs.get('name', '<no name>')}")

# For each record set, show its fields and their @ids
for rs in record_sets:
    print(f"\nFields for Record Set '@id': {rs['@id']}")
    if 'fields' in rs and rs['fields']:
        for field in rs['fields']:
            print(f"  - @id: {field['@id']}, name: {field.get('name', '<no name>')}, dataType: {field.get('dataType','')}" )
    else:
        print("  (No fields listed)")

## 3. Data Extraction
Extract data from all available record sets into DataFrames. Use the record set and field `@id`s from the overview step.

In [ ]:
# Collect and extract all record sets into pandas DataFrames, referenced by @id
dataframes = {}
list_of_record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in list_of_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: '{record_set_id}'")

# Show columns for the first record set (or choose one that's most descriptive)
example_rs_id = list_of_record_set_ids[0] if list_of_record_set_ids else None
if example_rs_id:
    print(f"\nColumns in record set '@id': {example_rs_id}")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps, such as filtering records, normalizing numeric fields, and grouping by key attributes. All operations reference fields by their `@id`.

We'll demonstrate EDA on a numeric field and a group-by field. Update these field @ids as appropriate for your analysis.

In [ ]:
# Example: pick a record set and numeric field (@id) for demonstration
# Below, update with an actual numeric field (as found in section 2!)

record_set_id = example_rs_id # or another @id as needed
df = dataframes[record_set_id] if record_set_id else None

# Replace with the actual field @id for a numeric variable (from overview step)
numeric_field_id = None
group_field_id = None

# Suggest likely candidates for numeric fields and group fields
if df is not None:
    # Attempt to auto-detect a float/integer column, otherwise use user input
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    group_candidates = [col for col in df.columns if df[col].nunique() < len(df)//3 and df[col].dtype==object]
    if group_candidates:
        group_field_id = group_candidates[0]

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Optional group by categorical field
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No numeric field automatically detected. Please set 'numeric_field_id' to a field with numeric data for your record set.")

## 5. Visualization
Visualize distribution or relationships between fields. You may need to select numeric or categorical fields, referencing them by `@id` as before.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if a DataFrame and numeric field are defined
if df is not None and numeric_field_id:
    # Basic histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '@id': {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a group field is available, show violinplot
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.violinplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore a complex Croissant-described dataset using `mlcroissant`. We programmatically referenced record sets and fields by their `@id`, extracted data into pandas DataFrames, and performed exploratory data analysis. For advanced analysis, consult the dataset's schema documentation and expand EDA and visualization sections as appropriate for your scientific or ML workflow.